# Diferential expression analysis by Sex

In [27]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd
from anndata import AnnData
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats


In [53]:
experiment = 'RNAseq_abundances_adjusted_combat_inmose'
age = 'old'
comparison = 'male.vs.female'

In [54]:
adata = pd.read_csv(f'/home/amore/work/data/{experiment}_gene_symbol_expression.csv', index_col=0)
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,12741169,0,0,0,0,0,0,2773,0,91.0
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,4248288,0,0,0,0,0,0,1,0,69.0
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,10497182,0,0,0,0,0,0,1614,0,83.0
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,10506926,0,0,0,0,0,0,795,0,71.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,23486240,0,0,0,0,0,0,0,5351828,27.5
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,34582470,0,0,0,1956002,0,0,0,0,27.5
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,81916604,0,1537700,0,824055,0,1040940,12712840,5952181,27.5


In [55]:
metadata_file = '/home/amore/work/data/All_rna_samples_metadata.csv'
metadata_df = pd.read_csv(metadata_file, index_col=0)
metadata_df

,Age,Status,Experiment,Sex
Run,,,,
SRR13758984,91.0,Sarcopenia,GSE167186,NaN
SRR13758985,86.0,Healthy,GSE167186,male
SRR13758986,69.0,Healthy,GSE167186,male
SRR13758987,83.0,Sarcopenia,GSE167186,NaN
SRR13758988,71.0,UNCLASSIFIED,GSE167186,NaN
...,...,...,...,...
SRR1555214,27.5,untrained,GSE60590,male
SRR1555215,27.5,untrained,GSE60590,male
SRR1555216,27.5,trained,GSE60590,male


In [56]:
metadata_df.drop(columns=['Age','Status','Experiment'], inplace=True)
metadata_df

,Sex
Run,
SRR13758984,NaN
SRR13758985,male
SRR13758986,male
SRR13758987,NaN
SRR13758988,NaN
...,...
SRR1555214,male
SRR1555215,male
SRR1555216,male


In [57]:
adata.head(2)

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,12741169,0,0,0,0,0,0,2773,0,91.0
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0


In [58]:
adata['Sex'] = metadata_df['Sex']
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,0,0,0,0,0,0,2773,0,91.0,NaN
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,0,0,0,0,0,0,1328,0,86.0,male
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,0,0,0,0,0,0,1,0,69.0,male
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,0,0,0,0,0,0,1614,0,83.0,NaN
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,0,0,0,0,0,0,795,0,71.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,0,0,0,0,0,0,0,5351828,27.5,male
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,0,0,0,1956002,0,0,0,0,27.5,male
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,0,1537700,0,824055,0,1040940,12712840,5952181,27.5,male


In [59]:
adata.loc[adata['Sex'].isna(), 'Sex'] = 'male'

In [60]:
adata.sort_values(by='Sex')

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13388751,222127430,406774,51853664,228028834,627791700,129589716,234481415,308033853,193496385,147703433,...,0,8000000,8097950,0,6663300,0,589174,13519400,80.0,female
SRR1555186,54149336,450349,55709971,264208,19333528,1,134109630,28594064,38540911,30889758,...,0,0,0,0,0,0,0,2263800,26.4,female
SRR13388736,253613257,198504,227481652,10651308,182165549,1,372074493,303323305,207882783,37430378,...,0,5000000,26685800,3346645,0,0,1889847,15203500,35.0,female
SRR1555180,28155151,0,10923412,8565773,869736,9817,85508014,34630306,41410065,24483270,...,0,0,0,0,0,0,0,0,26.4,female
SRR12021930,1121508954,1,2815570925,4300,99031446,1,3537198789,726219764,1946342166,5605136489,...,0,111000000,51652600,562889,48393800,0,314106732,471759000,83.0,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR8882187,260348025,6617,449953976,286,59320409,1,572959824,228953407,352765191,185557313,...,0,11000000,0,4797133,5000000,0,14879,19538180,67.0,male
SRR8882189,153466558,0,324063117,1,17620566,1,837300176,98261542,288671156,48668484,...,0,2000000,4000000,0,0,0,138749,3794310,72.0,male
SRR8882191,96204612,0,293431233,123715,15723598,1,499703693,108312058,245043098,228516317,...,0,0,1698520,3495605,0,2151350,193576,20604300,81.0,male


In [61]:
#----------------------------------------

In [62]:
age

'old'

In [63]:
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,0,0,0,0,0,0,2773,0,91.0,male
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,0,0,0,0,0,0,1328,0,86.0,male
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,0,0,0,0,0,0,1,0,69.0,male
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,0,0,0,0,0,0,1614,0,83.0,male
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,0,0,0,0,0,0,795,0,71.0,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,0,0,0,0,0,0,0,5351828,27.5,male
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,0,0,0,1956002,0,0,0,0,27.5,male
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,0,1537700,0,824055,0,1040940,12712840,5952181,27.5,male


In [64]:
age_series = adata['Age']
age_series = age_series.apply(lambda age: 'young' if age < 35 else ('old' if age >= 65 else 'middle'))
adata['Age'] = age_series


In [65]:
adata = adata[adata['Age']==age]
adata.head(2)

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,0,0,0,0,0,0,2773,0,old,male
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,0,0,0,0,0,0,1328,0,old,male


In [66]:
adata.sort_values(by='Sex')

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR12021930,1121508954,1,2815570925,4300,99031446,1,3537198789,726219764,1946342166,5605136489,...,0,111000000,51652600,562889,48393800,0,314106732,471759000,old,female
SRR13388751,222127430,406774,51853664,228028834,627791700,129589716,234481415,308033853,193496385,147703433,...,0,8000000,8097950,0,6663300,0,589174,13519400,old,female
SRR13388750,189994903,0,434876077,1,24630205,1,974291807,138250503,374622794,103958428,...,0,2000000,4000000,0,0,0,451335,3794310,old,female
SRR8882188,170874691,0,261210728,1,35393916,81816,455213853,120326554,87875461,1572902,...,0,1000000,0,0,0,0,0,31675500,old,female
SRR8882190,170783071,336775,34659416,140152108,627825800,113582187,206868189,251309627,153349433,104779728,...,0,8000000,8097950,0,6663300,0,224877,13519400,old,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR13759042,6580969,0,21831505,0,10915244,0,14823212,8677090,7797037,20028310,...,0,0,0,0,0,0,37642,0,old,male
SRR13759041,3155149,0,0,0,0,0,15316538,5069058,8475104,15094544,...,0,0,0,0,0,0,16,0,old,male
SRR13759040,0,0,9133851,0,0,0,11584223,7578506,10932166,0,...,0,0,0,0,0,0,1793,0,old,male


In [67]:
#adata=adata.astype(int)
#adata.isnull().sum().any()

In [68]:
adata=adata.drop('Age',axis=1)
sex= adata['Sex']
adata=adata.drop('Sex',axis=1)

In [69]:

andata=AnnData(adata.to_numpy(), obs=pd.DataFrame(sex))
andata.obs_names = adata.index
andata.var_names = adata.columns


In [70]:
#adata = andata #AnnData(adata.to_numpy(), dtype=np.int32)
andata.var_names_make_unique()
andata

AnnData object with n_obs × n_vars = 137 × 34327
    obs: 'Sex'

In [71]:
# Process treatment information
#adata.obs['condition'] = age_series.apply(lambda age: 'young' if age < 35 else ('old' if age >= 65 else 'middle'))
andata.obs['condition'] = sex

In [72]:
andata.obs['condition']

Sample
SRR13758984      male
SRR13758985      male
SRR13758986      male
SRR13758987      male
SRR13758988      male
                ...  
SRR12604201      male
SRR12604202      male
SRR12604203      male
SRR12021929      male
SRR12021930    female
Name: condition, Length: 137, dtype: object

In [73]:
# Obtain genes that pass the thresholds
genes = dc.filter_by_expr(andata, group='condition', min_count=10, min_total_count=15, large_n=1, min_prop=1)

# Filter by these genes
andata = andata[:, genes].copy()
andata

AnnData object with n_obs × n_vars = 137 × 32885
    obs: 'Sex', 'condition'

In [74]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=andata,
    design_factors='condition',
    refit_cooks=True,
    inference=inference,
)


In [75]:
dds.deseq2()

Fitting size factors...
... done in 0.09 seconds.

Fitting dispersions...
... done in 18.96 seconds.

Fitting dispersion trend curve...
... done in 0.59 seconds.

Fitting MAP dispersions...
... done in 26.59 seconds.

Fitting LFCs...
... done in 35.70 seconds.

Calculating cook's distance...
... done in 0.35 seconds.

Replacing 3463 outlier genes.

Fitting dispersions...
... done in 1.44 seconds.

Fitting MAP dispersions...
... done in 1.55 seconds.

Fitting LFCs...
... done in 4.21 seconds.



In [76]:
def get_DDS(younger_group, older_group, sex,  save=True):
    comparison = f'{younger_group}.vs.{older_group}'
    stat_res = DeseqStats(
        dds,
        contrast=["condition", younger_group, older_group],
        inference=inference
    )
    stat_res.summary()
    results_df = stat_res.results_df
    if not results_df is None:
        results_df.to_csv(f'/home/amore/work/data/{experiment}_{comparison}_{sex}_DDS.csv', header=True)
    return results_df

In [77]:
get_DDS(younger_group="male", older_group="female", sex=age)

Running Wald tests...
... done in 2.30 seconds.



Log2 fold change & Wald test p-value: condition male vs female
                    baseMean  log2FoldChange       lfcSE      stat    pvalue  \
TSPAN6          5.162900e+07       -0.984723    0.408214 -2.412270  0.015854   
TNMD            2.386928e+05        0.860423    5.185300  0.165935  0.868208   
DPM1            7.657853e+07        0.102579    0.482704  0.212508  0.831710   
SCYL3           1.352448e+07       -2.512500    2.536204 -0.990654  0.321855   
FIRRM           3.522525e+07       -3.052656    1.588645 -1.921547  0.054663   
...                      ...             ...         ...       ...       ...   
H2BK1           1.052180e+05        5.012685    3.081206  1.626858  0.103767   
OR1Q1BP         1.974759e+05       -4.709587    7.693064 -0.612186  0.540415   
Unnamed: 34325  3.783219e+03       29.501703  100.101563  0.294718  0.768210   
Unnamed: 34326  3.661128e+06        0.860422    2.332794  0.368838  0.712249   
TBCEL-TECTA     1.842840e+06       -2.752223    2.395129 

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.162900e+07,-0.984723,0.408214,-2.412270,0.015854,0.373084
TNMD,2.386928e+05,0.860423,5.185300,0.165935,0.868208,0.922530
DPM1,7.657853e+07,0.102579,0.482704,0.212508,0.831710,0.900361
SCYL3,1.352448e+07,-2.512500,2.536204,-0.990654,0.321855,0.751959
FIRRM,3.522525e+07,-3.052656,1.588645,-1.921547,0.054663,0.705512
...,...,...,...,...,...,...
H2BK1,1.052180e+05,5.012685,3.081206,1.626858,0.103767,0.751959
OR1Q1BP,1.974759e+05,-4.709587,7.693064,-0.612186,0.540415,0.752288
Unnamed: 34325,3.783219e+03,29.501703,100.101563,0.294718,0.768210,0.862487
Unnamed: 34326,3.661128e+06,0.860422,2.332794,0.368838,0.712249,0.828167
